# Phase 2A — Generate Full 733-Node LP Files for CPLEX (Full Licence)

**Project:** Flood-Risk-Weighted Emergency Shelter Optimization, Pinellas County, FL  
**Authors:** Christopher Atta Amponsah, Towfiqul Islam Khan, Chy Mansura Mehrun Mumu, Joni Downs  
**Institution:** School of Geosciences, University of South Florida  

---

## Purpose

This notebook generates LP (Linear Program) files for the **full, unaggregated p-median problem**
using all **733 census block groups** as demand nodes and **25 candidate shelter sites** as facilities.

Unlike the aggregated version (`phase2a_cplex_lp_aggregated.ipynb`), which reduces the problem
to 38 clusters to satisfy the CPLEX Community Edition variable limit (≤1,000 variables),
this notebook generates the **exact full-resolution problem** requiring the **CPLEX full licence**.

| | Aggregated (Community Ed.) | Full (This notebook) |
|---|---|---|
| Demand nodes | 38 clusters | 733 block groups |
| Decision variables | 975 | 18,325 |
| Constraints | 989 | 19,059 |
| Requires full licence | No | **Yes** |
| Aggregation bias | Present | None |

## Inputs Required

| File | Description | Produced by |
|---|---|---|
| `demand_nodes.csv` | 733 block groups with population, flood-risk weights, lat/lon | `phase1b_flood_demand_od_matrix.ipynb` |
| `distance_matrix_network.csv` | 733×25 road-network travel time matrix (minutes) | `phase1b_flood_demand_od_matrix.ipynb` |

## Outputs

| File | Description |
|---|---|
| `pmedian_full_p5.lp` | LP file for p=5 (5 open shelters) |
| `pmedian_full_p8.lp` | LP file for p=8 (8 open shelters) |
| `pmedian_full_p10.lp` | LP file for p=10 (10 open shelters) |

## CPLEX Instructions (for Dr. Joni Downs)

After running this notebook, open the CPLEX Interactive Optimizer and for each LP file:
```
read <path>\pmedian_full_p5.lp
optimize
write <path>\solution_full_p5.sol
```
Repeat for p8 and p10. Then run `phase2a_cplex_solution_parser.ipynb` to convert
the `.sol` files into assignment CSV tables.

---

## p-Median Formulation

**Objective:** Minimise total flood-risk-weighted travel burden

$$\min \sum_{i \in I} \sum_{j \in J} w_i \cdot d_{ij} \cdot y_{ij}$$

**Subject to:**
- $\sum_{j} x_j = p$ — exactly p shelters open
- $\sum_{j} y_{ij} = 1 \; \forall i$ — each demand node assigned to exactly one shelter
- $y_{ij} \leq x_j \; \forall i,j$ — only assign to open shelters
- $x_j, y_{ij} \in \{0,1\}$ — binary decision variables

**Notation:**
- $w_i$ = flood-risk-weighted demand at block group $i$ (population × FEMA multiplier φ)
- $d_{ij}$ = road-network travel time from block group $i$ to shelter $j$ (minutes)
- $x_j = 1$ if shelter $j$ is opened
- $y_{ij} = 1$ if block group $i$ is assigned to shelter $j$

In [ ]:
# =============================================================================
# CELL 1 — CONFIGURATION
# Update the file paths below to match your local directory structure.
# All other cells can be run as-is.
# =============================================================================

import pandas as pd
import numpy as np
import os

# ── INPUT FILE PATHS ──────────────────────────────────────────────────────────
# demand_nodes.csv: output of phase1b_flood_demand_od_matrix.ipynb
# Contains columns: GEOID_JOIN, LAT, LON, POPULATION, WEIGHTED_DEMAND,
#                   TIER_LABEL, MULTIPLIER
DEMAND_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\demand_nodes.csv"

# distance_matrix_network.csv: 733×25 OD travel-time matrix (minutes)
# Rows = block group GEOID_JOIN, Columns = shelter IDs (S0..S24)
MATRIX_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\distance_matrix_network.csv"

# ── OUTPUT DIRECTORY ──────────────────────────────────────────────────────────
# LP files will be written here.
# Create the folder if it doesn't exist.
OUTPUT_DIR = r"D:\GIS_Seminar_Project\CPLEX_LP_Full"

# ── SHELTER SCENARIOS ─────────────────────────────────────────────────────────
# p = number of shelters to open.
# Three scenarios: low (5), moderate (8), high (10) resource allocation.
P_VALUES = [5, 8, 10]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# CELL 2 — LOAD AND VALIDATE INPUT DATA
# =============================================================================

# ── Load demand nodes ─────────────────────────────────────────────────────────
# GEOID_JOIN must be read as string to preserve leading zeros.
demand = pd.read_csv(DEMAND_CSV, dtype={"GEOID_JOIN": str})
demand["GEOID_JOIN"] = demand["GEOID_JOIN"].str.replace(r"\.0$", "", regex=True)

# ── Load travel-time matrix ───────────────────────────────────────────────────
# Rows = block groups (GEOID_JOIN), Columns = shelter sites (S0..S24)
matrix = pd.read_csv(MATRIX_CSV, index_col=0, dtype={0: str})
matrix.index = matrix.index.astype(str).str.replace(r"\.0$", "", regex=True)

# ── Align demand nodes with matrix rows ──────────────────────────────────────
# Keep only block groups that appear in both files (should be all 733).
common_ids = demand["GEOID_JOIN"][demand["GEOID_JOIN"].isin(matrix.index)]
demand = demand[demand["GEOID_JOIN"].isin(common_ids)].reset_index(drop=True)
matrix = matrix.loc[demand["GEOID_JOIN"]]

# ── Extract arrays ────────────────────────────────────────────────────────────
# W: flood-risk-weighted demand for each block group (length = n_I)
# D: travel-time matrix, shape (n_I, n_J) — minutes
W = demand["WEIGHTED_DEMAND"].values.astype(float)   # shape: (733,)
D = matrix.values.astype(float)                       # shape: (733, 25)

n_I = len(W)    # number of demand nodes (block groups)
n_J = D.shape[1]  # number of candidate shelters

# ── Problem size summary ──────────────────────────────────────────────────────
n_vars        = n_J + n_I * n_J          # x_j + y_ij variables
n_constraints = 1 + n_I + n_I * n_J     # open_p + assign_i + link_ij constraints

print(f"Demand nodes (block groups): {n_I}")
print(f"Candidate shelters:          {n_J}")
print(f"Total variables:             {n_vars:,}  (CPLEX Community limit: 1,000)")
print(f"Total constraints:           {n_constraints:,}  (CPLEX Community limit: 1,000)")
print(f"")
print(f"Total flood-risk-weighted demand: {W.sum():,.1f} person-units")
print(f"Travel-time range: {D.min():.2f} – {D.max():.2f} minutes")
print(f"Travel-time mean:  {D.mean():.2f} minutes")
print()
print("NOTE: This problem REQUIRES the CPLEX full licence.")
print("      The Community Edition cannot solve this (variable limit exceeded).")

In [ ]:
# =============================================================================
# CELL 3 — LP FILE WRITER FUNCTION
#
# Writes a valid CPLEX LP-format file for the p-median problem.
#
# Variable naming convention:
#   x_j       = 1 if shelter j is opened (j = 0..24)
#   y_i_j     = 1 if block group i is assigned to shelter j
#
# Constraint naming convention:
#   open_p    = exactly p shelters must be opened
#   assign_i  = block group i assigned to exactly one shelter
#   link_i_j  = block group i cannot be assigned to closed shelter j
# =============================================================================

def write_full_lp_file(W, D, p, shelter_names, demand_ids, filepath):
    """
    Write a CPLEX LP file for the full (unaggregated) p-median problem.

    Parameters
    ----------
    W            : array, shape (n_I,)    — flood-risk-weighted demand per block group
    D            : array, shape (n_I,n_J) — travel times from block groups to shelters
    p            : int                    — number of shelters to open
    shelter_names: list of str            — column names from distance matrix (S0..S24)
    demand_ids   : list of str            — GEOID_JOIN for each block group
    filepath     : str                    — output path for .lp file
    """
    n_I, n_J = D.shape
    lines = []

    # ── File header ───────────────────────────────────────────────────────────
    lines.append(f"\\Problem name: pmedian_full_p{p}")
    lines.append(f"\\")
    lines.append(f"\\  Flood-Risk-Weighted p-Median: Full 733-Node Problem")
    lines.append(f"\\  Pinellas County Emergency Shelter Placement")
    lines.append(f"\\  Amponsah, Khan, Mumu, Downs — University of South Florida")
    lines.append(f"\\")
    lines.append(f"\\  Demand nodes (I): {n_I} census block groups")
    lines.append(f"\\  Candidate sites (J): {n_J} Non-Evacuation Zone shelters")
    lines.append(f"\\  Open shelters (p): {p}")
    lines.append(f"\\  Decision variables: {n_J + n_I*n_J:,}")
    lines.append(f"\\  Constraints:        {1 + n_I + n_I*n_J:,}")
    lines.append(f"\\")
    lines.append(f"\\  Objective: minimise sum_i sum_j w_i * d_ij * y_ij")
    lines.append(f"\\  where w_i = flood-risk-weighted demand (population x FEMA phi)")
    lines.append(f"\\        d_ij = road-network travel time in minutes")
    lines.append("")

    # ── Objective function ────────────────────────────────────────────────────
    # Minimise: sum over all (i,j) of w_i * d_ij * y_i_j
    lines.append("Minimize")
    lines.append(" obj:")

    terms = []
    for i in range(n_I):
        for j in range(n_J):
            coeff = W[i] * D[i, j]
            if coeff > 0:
                terms.append(f"{coeff:.6f} y_{i}_{j}")

    # Write objective terms, 5 per line for readability
    for k in range(0, len(terms), 5):
        chunk = terms[k:k+5]
        prefix = "  " if k == 0 else "  + "
        lines.append(prefix + " + ".join(chunk))
    lines.append("")

    # ── Constraints ───────────────────────────────────────────────────────────
    lines.append("Subject To")

    # Constraint 1: exactly p shelters must be opened
    # sum_j x_j = p
    lines.append(f" open_p: {' + '.join(f'x_{j}' for j in range(n_J))} = {p}")

    # Constraint 2: each block group assigned to exactly one shelter
    # sum_j y_i_j = 1  for all i
    for i in range(n_I):
        lhs = " + ".join(f"y_{i}_{j}" for j in range(n_J))
        lines.append(f" assign_{i}: {lhs} = 1")

    # Constraint 3: block group i can only be assigned to shelter j if j is open
    # y_i_j - x_j <= 0  for all i, j
    for i in range(n_I):
        for j in range(n_J):
            lines.append(f" link_{i}_{j}: y_{i}_{j} - x_{j} <= 0")
    lines.append("")

    # ── Bounds ────────────────────────────────────────────────────────────────
    # All variables are binary (0 or 1), but CPLEX LP format requires
    # explicit bounds for variables listed in the Binary section.
    lines.append("Bounds")
    for j in range(n_J):
        lines.append(f" 0 <= x_{j} <= 1")
    for i in range(n_I):
        for j in range(n_J):
            lines.append(f" 0 <= y_{i}_{j} <= 1")
    lines.append("")

    # ── Binary declarations ───────────────────────────────────────────────────
    lines.append("Binary")
    # x_j: shelter open/closed indicators
    lines.append(" " + " ".join(f"x_{j}" for j in range(n_J)))
    # y_i_j: assignment indicators (10 per line for readability)
    y_vars = [f"y_{i}_{j}" for i in range(n_I) for j in range(n_J)]
    for k in range(0, len(y_vars), 10):
        lines.append(" " + " ".join(y_vars[k:k+10]))
    lines.append("")
    lines.append("End")

    # ── Write to disk ─────────────────────────────────────────────────────────
    with open(filepath, "w") as f:
        f.write("\n".join(lines))

    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  Written: {os.path.basename(filepath)}")
    print(f"           {n_J + n_I*n_J:,} variables  |  "
          f"{1 + n_I + n_I*n_J:,} constraints  |  {size_mb:.1f} MB")

print("LP writer function defined — ready to generate files.")

In [ ]:
# =============================================================================
# CELL 4 — GENERATE LP FILES
#
# Generates one LP file per shelter scenario (p = 5, 8, 10).
# Each file encodes the complete full-resolution p-median problem
# with all 733 block groups as demand nodes.
# =============================================================================

shelter_names = list(matrix.columns)   # ['S0_Clearwater_Fund', 'S1_Palm_Harbor_Mid', ...]
demand_ids    = list(demand["GEOID_JOIN"])  # GEOID_JOIN for each block group

print(f"Generating full 733-node LP files for p ∈ {P_VALUES} ...")
print(f"Output directory: {OUTPUT_DIR}")
print()

for p in P_VALUES:
    filepath = os.path.join(OUTPUT_DIR, f"pmedian_full_p{p}.lp")
    write_full_lp_file(W, D, p, shelter_names, demand_ids, filepath)
    print()

print("All LP files generated successfully.")

In [ ]:
# =============================================================================
# CELL 5 — SAVE METADATA FOR SOLUTION PARSING
#
# Saves a metadata CSV that maps shelter column indices (0..24) back to
# their names and IDs. This is needed by the solution parser notebook
# (phase2a_cplex_solution_parser.ipynb) to reconstruct shelter assignments.
# =============================================================================

# Shelter metadata: index → name mapping
shelter_meta = pd.DataFrame({
    "SHELTER_IDX":  list(range(n_J)),
    "SHELTER_NAME": shelter_names,
    "VAR_NAME":     [f"x_{j}" for j in range(n_J)],
})
shelter_meta_path = os.path.join(OUTPUT_DIR, "shelter_metadata.csv")
shelter_meta.to_csv(shelter_meta_path, index=False)
print(f"Saved: shelter_metadata.csv")
print(shelter_meta.to_string(index=False))

# Demand node metadata: GEOID_JOIN, population, weighted demand, FEMA tier
demand_meta_cols = ["GEOID_JOIN", "LAT", "LON", "POPULATION",
                    "WEIGHTED_DEMAND"]
if "TIER_LABEL" in demand.columns:
    demand_meta_cols.append("TIER_LABEL")
if "MULTIPLIER" in demand.columns:
    demand_meta_cols.append("MULTIPLIER")

demand_meta = demand[demand_meta_cols].copy()
# Add the row index (0..732) used in y_i_j variable names
demand_meta.insert(0, "NODE_IDX", range(len(demand_meta)))
demand_meta_path = os.path.join(OUTPUT_DIR, "demand_node_metadata.csv")
demand_meta.to_csv(demand_meta_path, index=False)
print(f"Saved: demand_node_metadata.csv  ({len(demand_meta)} rows)")

---

## CPLEX Interactive Optimizer Instructions

Once the LP files are generated, open the **CPLEX Interactive Optimizer** and run the
following commands for each scenario. Replace `<OUTPUT_DIR>` with your actual path.

### p = 5 (low resource scenario)
```
read <OUTPUT_DIR>\pmedian_full_p5.lp
optimize
write <OUTPUT_DIR>\solution_full_p5.sol
```

### p = 8 (moderate resource scenario — recommended optimum)
```
read <OUTPUT_DIR>\pmedian_full_p8.lp
optimize
write <OUTPUT_DIR>\solution_full_p8.sol
```

### p = 10 (high resource scenario)
```
read <OUTPUT_DIR>\pmedian_full_p10.lp
optimize
write <OUTPUT_DIR>\solution_full_p10.sol
```

After solving all three, share the three `.sol` files and run
`phase2a_cplex_solution_parser.ipynb` to produce assignment tables.

---

## Expected Results (for verification)

The PuLP/CBC exact solver (open-source, run on same full 733-node problem) produced:

| p | PuLP/CBC Objective (person·min) | Expected CPLEX range |
|---|---|---|
| 5  | 4,184,071 | ≈ 4,184,071 (exact match expected) |
| 8  | 3,774,238 | ≈ 3,774,238 (exact match expected) |
| 10 | 3,641,580 | ≈ 3,641,580 (exact match expected) |

Both solvers are solving the **identical problem instance**, so solutions should agree.
Any difference would indicate a numerical tolerance issue or time limit.
